# OceanGuard AI — xView3 Fine-Tuning: All Training Runs

This notebook documents all fine-tuning experiments on the YOLO11n model, evaluating performance on real Sentinel-1 imagery from the xView3 dataset.

## Context
- **Baseline model:** `ml/models/best.pt` — YOLO11n fine-tuned on HRSID (SAR dataset, 0.5–3 m pixels), mAP@50=0.838 on HRSID test set.
- **Problem:** HRSID metrics do not transfer to real Sentinel-1 (10 m pixels). Baseline recall on xView3 was only 30.9%.
- **Goal:** Fine-tune from `best.pt` on xView3 train scenes + validate on held-out xView3 val scenes. Accept only if: (a) xView3 recall improves, (b) HRSID mAP drops ≤0.02.
- **Accepted model:** Run E → `ml/models/finetuned_runE.pt`

## Run Summary
| Run | Scenes | Epochs | xView3 Recall | HRSID mAP after | HRSID drop | Accept |
|-----|--------|--------|--------------|-----------------|------------|--------|
| B   | 5 xView3 val  | 20     | 41.9%        | 0.884           | 0.029      | ❌ |
| C   | 5 xView3 val  | 20     | 39.6%        | 0.887           | 0.026      | ❌ |
| D   | 5 xView3 val  | 20     | 41.9%        | 0.884           | 0.029      | ❌ |
| **E** | **8 xView3 train** | **20** | **42.2%** | **0.906** | **0.007** | **✅** |

In [ ]:
import json
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

RUNS_DIR = pathlib.Path('../training/runs')
RUNS = ['run_B', 'run_C', 'run_D', 'run_E']

def load_run(run_name):
    p = RUNS_DIR / run_name
    return {
        'summary': json.loads((p / 'summary.json').read_text()),
        'base_eval': json.loads((p / 'xview3_eval_base.json').read_text()),
        'ft_eval': json.loads((p / 'xview3_eval_finetuned.json').read_text()),
        'results': pd.read_csv(p / 'results.csv'),
    }

data = {r: load_run(r) for r in RUNS}
print('Loaded runs:', list(data.keys()))

## 1. Overall Recall and HRSID mAP Comparison

In [ ]:
summary_rows = []
for run, d in data.items():
    s = d['summary']
    summary_rows.append({
        'Run': run,
        'xView3 Recall (base)': f"{s['xview3_recall_base']:.1%}",
        'xView3 Recall (fine-tuned)': f"{s['xview3_recall_finetuned']:.1%}",
        'xView3 Precision (ft)': f"{s['xview3_precision_finetuned']:.1%}",
        'HRSID mAP50 (before)': f"{s['hrsid_map50_base']:.3f}",
        'HRSID mAP50 (after)': f"{s['hrsid_map50_finetuned']:.3f}",
        'HRSID drop': f"{s['hrsid_map50_base'] - s['hrsid_map50_finetuned']:.3f}",
        'Accept': '✅' if s['accept'] else '❌',
    })

df = pd.DataFrame(summary_rows).set_index('Run')
df

## 2. Size-Binned Recall: Baseline vs Best (Run E)

In [ ]:
def extract_bins(eval_data):
    rows = []
    for b in eval_data['bins']:
        rows.append({
            'label': b['label'],
            'truth': b['truth'],
            'recall': b['recall'],
            'ci_lo': b['recall_ci95'][0] if b['recall_ci95'] else None,
            'ci_hi': b['recall_ci95'][1] if b['recall_ci95'] else None,
        })
    return pd.DataFrame(rows)

base_bins  = extract_bins(data['run_E']['base_eval'])
ft_bins_E  = extract_bins(data['run_E']['ft_eval'])
ft_bins_B  = extract_bins(data['run_B']['ft_eval'])

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
labels = [b for b in base_bins['label'] if base_bins.loc[base_bins['label']==b,'truth'].values[0] > 0]

x = np.arange(len(labels))
w = 0.28

def bar_with_ci(ax, xs, df, color, label, offset):
    vals = [df.loc[df['label']==l,'recall'].values[0] or 0 for l in labels]
    cilo = [df.loc[df['label']==l,'ci_lo'].values[0] or 0 for l in labels]
    cihi = [df.loc[df['label']==l,'ci_hi'].values[0] or 0 for l in labels]
    yerr_lo = [v - lo for v, lo in zip(vals, cilo)]
    yerr_hi = [hi - v for v, hi in zip(vals, cihi)]
    ax.bar(xs + offset, vals, w, color=color, label=label, alpha=0.85)
    ax.errorbar(xs + offset, vals, yerr=[yerr_lo, yerr_hi],
                fmt='none', color='black', capsize=4, linewidth=1)

bar_with_ci(ax, x, base_bins, '#5b9bd5', 'Baseline best.pt', -w)
bar_with_ci(ax, x, ft_bins_B, '#ed7d31', 'Run B (rejected)', 0)
bar_with_ci(ax, x, ft_bins_E, '#70ad47', 'Run E — finetuned_runE.pt (accepted)', w)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Recall (95% Wilson CI)')
ax.set_title('Size-Binned Recall on xView3 Validation Scenes\n(match radius = 30 m, conf ≥ 0.25)')
ax.legend()
ax.set_ylim(0, 0.85)
ax.axhline(0, color='black', linewidth=0.5)

# Annotate truth counts
truth_counts = {l: int(base_bins.loc[base_bins['label']==l,'truth'].values[0]) for l in labels}
for i, l in enumerate(labels):
    ax.text(x[i], -0.06, f"n={truth_counts[l]}", ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../training/recall_by_size.png', dpi=150)
plt.show()
print('Saved to ml/training/recall_by_size.png')

## 3. Training Curves — Run E (accepted)

In [ ]:
results_E = data['run_E']['results']
# Strip whitespace from column names
results_E.columns = results_E.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(results_E['epoch'], results_E['metrics/mAP50(B)'], 'b-o', markersize=3)
axes[0].set_title('mAP@50 (validation)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('mAP@50')
axes[0].axhline(0.838, color='red', linestyle='--', label='HRSID baseline (best.pt)')
axes[0].legend()

loss_cols = [c for c in results_E.columns if 'train' in c and 'loss' in c]
for col in loss_cols:
    axes[1].plot(results_E['epoch'], results_E[col], label=col.replace('train/',''))
axes[1].set_title('Training Loss')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)

axes[2].plot(results_E['epoch'], results_E['metrics/precision(B)'], label='Precision', color='green')
axes[2].plot(results_E['epoch'], results_E['metrics/recall(B)'], label='Recall', color='orange')
axes[2].set_title('Precision / Recall (validation)')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.suptitle('Run E Training Curves (20 epochs, 8 xView3 train scenes, CPU)')
plt.tight_layout()
plt.savefig('../training/run_E_curves.png', dpi=150)
plt.show()

## 4. FP/100 km² Trade-off

In [ ]:
rows = []
for run, d in data.items():
    rows.append({
        'Run': run,
        'Model': 'baseline' if run == 'run_B' else run,
        'Recall': d['ft_eval']['overall_recall'],
        'FP/100km²': d['ft_eval']['false_positives_per_100km2'],
        'Accept': d['summary']['accept'],
    })
# Add baseline
rows.insert(0, {
    'Run': 'baseline',
    'Model': 'best.pt (no fine-tune)',
    'Recall': data['run_B']['base_eval']['overall_recall'],
    'FP/100km²': data['run_B']['base_eval']['false_positives_per_100km2'],
    'Accept': None,
})

fig, ax = plt.subplots(figsize=(8, 5))
for row in rows:
    color = '#70ad47' if row['Accept'] else ('#5b9bd5' if row['Accept'] is None else '#ed7d31')
    marker = '*' if row['Accept'] else ('o' if row['Accept'] is None else 'x')
    ax.scatter(row['FP/100km²'], row['Recall'], s=120, color=color, marker=marker,
               label=f"{row['Run']} ({row['Recall']:.1%} recall)")
    ax.annotate(row['Run'], (row['FP/100km²'], row['Recall']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

ax.set_xlabel('False Positives per 100 km²')
ax.set_ylabel('Overall Recall (930 ground-truth vessels)')
ax.set_title('Recall vs FP Rate — All Runs')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('../training/recall_vs_fp.png', dpi=150)
plt.show()

## 5. Key Findings

### Baseline (best.pt on xView3)
- Overall recall: **30.9%** on 930 ground-truth vessels
- Small vessels (15–30 m): only **5%** recall — nearly invisible
- Medium vessels (30–60 m): only **9.5%** recall
- Large vessels (≥120 m): **65.4%** recall (model works on large ships)
- FP rate: **0.37 FP/100 km²** — quite good precision

### Run E — finetuned_runE.pt (accepted ✅)
- Overall recall: **42.2%** (+11.3 pp over baseline)
- Small vessels (15–30 m): **25%** (+20 pp, 5× improvement)
- Medium vessels (30–60 m): **43.7%** (+34 pp)
- Large vessels (≥120 m): **66.3%** (+0.9 pp, maintained)
- HRSID mAP dropped: 0.913 → 0.906 (**0.7% drop**, well within 2% gate)
- FP rate: **0.92 FP/100 km²** (2.5× increase — precision-recall trade-off)

### Why runs B, C, D were rejected
- Runs B and D: HRSID mAP dropped 0.029 (above the 0.02 gate)
- Run C: HRSID mAP dropped 0.026 (above gate)
- Run E succeeded because it trained on proper xView3 train-split scenes
  (not validation scenes), preserving the HRSID distribution better.

### Remaining limitations
- 0–15 m vessels: 0 ground-truth in the validation split (no measurement)
- FP/100 km² increased from 0.37 → 0.92 — still low but 2.5× worse
- Only 3 xView3 validation scenes used (930 vessels). Confidence intervals are wide
  for small-vessel bins. More scenes would narrow them.
- All training ran on CPU (Azure student sub has no GPU quota). Each run ≈ 1 hour.

## 6. How to Use the Accepted Model

```python
from ultralytics import YOLO

# Load the accepted fine-tuned model
model = YOLO('ml/models/finetuned_runE.pt')

# Run on a Sentinel-1 chip
results = model.predict('path/to/chip.png', conf=0.25, imgsz=640)
boxes = results[0].boxes  # xyxy, confidence, class
```

The YOLO service (`yolo-service/`) uses `ml/models/best.pt` by default.
To switch to the fine-tuned model, set `MODEL_PATH=ml/models/finetuned_runE.pt`
in `yolo-service/.env`.

⚠️ Do not use the fine-tuned model without noting the higher FP rate (0.92 vs 0.37 FP/100 km²).